# GRU

**Capítulo 5 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_recurrent-modern/gru.ipynb` · [Lección original](https://d2l.ai/chapter_recurrent-modern/gru.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Unidades recurrentes con compuertas (GRU)
<a id="sec_gru"></a>

A medida que las RNN y en particular la arquitectura LSTM ([Referencia sec_lstm](https://d2l.ai/chapter_recurrent-modern/lstm.html#sec-lstm)) ganaron rápidamente popularidad durante la década de 2010, varios investigadores comenzaron a experimentar con arquitecturas simplificadas con la esperanza de conservar la idea clave de incorporar un estado interno y mecanismos de gating multiplicativo, pero con el objetivo de acelerar el cálculo. La unidad recurrente cerrada (GRU) [Cho.Van-Merrienboer.Bahdanau.ea.2014](https://d2l.ai/chapter_references/zreferences.html) ofreció una versión aerodinámica de la célula de memoria LSTM que a menudo logra un rendimiento comparable pero con la ventaja de ser más rápido para calcular [Chung.Gulcehre.Cho.ea.2014](https://d2l.ai/chapter_references/zreferences.html).


In [ ]:
import torch
from torch import nn
from laboratorio import d2l

## Comcompuerta de reinicio y comcompuerta de actualización
Aquí, las tres puertas del LSTM son reemplazadas por dos: la *puerta de reset* y la *compuerta de actualización*. Al igual que con las LSTMs, a estas puertas se les dan activaciones sigmoide, forzando a sus valores a estar en el intervalo $(0, 1)$. Intuitivamente, la puerta de reset controla cuánto del estado anterior que todavía quisiéramos recordar. Del mismo modo, una compuerta de actualización nos permitiría controlar cuánto del nuevo estado es sólo una copia del estado antiguo.
[Referencia fig_gru_1](https://d2l.ai/chapter_recurrent-modern/gru.html#fig-gru-1) ilustra las entradas de ambas compuertas
las puertas de reinicio y actualización en una GRU, dada la entrada del paso de tiempo actual y el estado oculto del paso de tiempo anterior. Las salidas de las puertas están dadas por dos capas totalmente conectadas con una función de activación sigmoide.

![Cálculo de las compuertas de reinicio y actualización en una GRU.](../recursos/originales/gru-1.svg)
<a id="fig_gru_1"></a>

Matemáticamente, para un tiempo dado paso $t$, suponga que la entrada es un minibatch $\mathbf{X}_t \in \mathbb{R}^{n \times d}$ (número de ejemplos $=n$; número de entradas $=d$) y el estado oculto del paso de tiempo anterior es $\mathbf{H}_{t-1} \in \mathbb{R}^{n \times h}$ (número de unidades ocultas $=h$). A continuación, la compuerta de reinicio $\mathbf{R}_t \in \mathbb{R}^{n \times h}$ y la compuerta de actualización $\mathbf{Z}_t \in \mathbb{R}^{n \times h}$ se calculan de la siguiente manera:

$$
\begin{aligned}
\mathbf{R}_t = \sigma(\mathbf{X}_t \mathbf{W}_{\textrm{xr}} + \mathbf{H}_{t-1} \mathbf{W}_{\textrm{hr}} + \mathbf{b}_\textrm{r}),\\
\mathbf{Z}_t = \sigma(\mathbf{X}_t \mathbf{W}_{\textrm{xz}} + \mathbf{H}_{t-1} \mathbf{W}_{\textrm{hz}} + \mathbf{b}_\textrm{z}),
\end{aligned}
$$

donde $\mathbf{W}_{\textrm{xr}}, \mathbf{W}_{\textrm{xz}} \in \mathbb{R}^{d \times h}$ y $\mathbf{W}_{\textrm{hr}}, \mathbf{W}_{\textrm{hz}} \in \mathbb{R}^{h \times h}$ son parámetros de peso y $\mathbf{b}_\textrm{r}, \mathbf{b}_\textrm{z} \in \mathbb{R}^{1 \times h}$ son parámetros de sesgo.

## Estado oculto del candidato
A continuación, integramos la compuerta de reinicio $\mathbf{R}_t$ con el mecanismo de actualización regular en [Referencia rnn_h_with_state](https://d2l.ai/#rnn-h-with-state), lo que conduce al siguiente *candidate estado oculto* $\tilde{\mathbf{H}}_t \in \mathbb{R}^{n \times h}$ en el momento paso $t$:

$$\tilde{\mathbf{H}}_t = \tanh(\mathbf{X}_t \mathbf{W}_{\textrm{xh}} + \left(\mathbf{R}_t \odot \mathbf{H}_{t-1}\right) \mathbf{W}_{\textrm{hh}} + \mathbf{b}_\textrm{h}),$$

:eqlabel:`gru_tilde_H`

donde $\mathbf{W}_{\textrm{xh}} \in \mathbb{R}^{d \times h}$ y $\mathbf{W}_{\textrm{hh}} \in \mathbb{R}^{h \times h}$ son parámetros de peso, $\mathbf{b}_\textrm{h} \in \mathbb{R}^{1 \times h}$ es el sesgo, y el símbolo $\odot$ es el operador de producto Hadamard (elementwise). Aquí utilizamos una función de activación tanh.

El resultado es un *candidato*, ya que todavía tenemos que incorporar la acción de la compuerta de actualización. Comparando con [Referencia rnn_h_with_state](https://d2l.ai/#rnn-h-with-state), la influencia de los estados anteriores ahora se puede reducir con la multiplicación de elementos de $\mathbf{R}_t$ y $\mathbf{H}_{t-1}$ en [Referencia gru_tilde_H](https://d2l.ai/#gru-tilde-H). Siempre que las entradas en la compuerta de reinicio $\mathbf{R}_t$ están cerca de 1, recuperamos un RNN convencionales como el de [Referencia rnn_h_with_state](https://d2l.ai/#rnn-h-with-state). Para todas las entradas de la compuerta de reinicio $\mathbf{R}_t$ que están cerca de 0, el estado oculto candidato es el resultado de un MLP con $\mathbf{X}_t$ como entrada. Cualquier estado oculto preexistente es así *restablecido* a valores predeterminados.

[Referencia fig_gru_2](https://d2l.ai/chapter_recurrent-modern/gru.html#fig-gru-2) muestra el flujo de cálculo después de aplicar la comcompuerta de reinicio.

![Cálculo del estado oculto candidato en una GRU.](../recursos/originales/gru-2.svg)
<a id="fig_gru_2"></a>

## Estado oculto
Por último, tenemos que incorporar el efecto de la compuerta de actualización $\mathbf{Z}_t$. Esto determina la medida en que el nuevo estado oculto $\mathbf{H}_t \in \mathbb{R}^{n \times h}$ coincide con el estado antiguo $\mathbf{H}_{t-1}$ en comparación con lo mucho que se asemeja al nuevo estado candidato $\tilde{\mathbf{H}}_t$. La compuerta de actualización $\mathbf{Z}_t$ se puede utilizar para este propósito, simplemente tomando combinaciones convexas de elementos de $\mathbf{H}_{t-1}$ y $\tilde{\mathbf{H}}_t$. Esto conduce a la ecuación de actualización final para la GRU:

$$\mathbf{H}_t = \mathbf{Z}_t \odot \mathbf{H}_{t-1}  + (1 - \mathbf{Z}_t) \odot \tilde{\mathbf{H}}_t.$$

Cada vez que la compuerta de actualización $\mathbf{Z}_t$ está cerca de 1, simplemente retenemos el estado antiguo. En este caso la información de $\mathbf{X}_t$ es ignorada, omitiendo efectivamente el paso $t$ en la cadena de dependencia. En cambio, cuando $\mathbf{Z}_t$ está cerca de 0, el nuevo estado latente $\mathbf{H}_t$ se acerca al estado latente candidato $\tilde{\mathbf{H}}_t$.
[Referencia fig_gru_3](https://d2l.ai/chapter_recurrent-modern/gru.html#fig-gru-3) muestra el flujo de cálculo cuando actúa la comcompuerta de actualización.

![Cálculo del estado oculto en una GRU.](../recursos/originales/gru-3.svg)
<a id="fig_gru_3"></a>

En resumen, las GRUs tienen dos características distintivas:

* Reset gates ayuda a capturar dependencias a corto plazo en secuencias.
* Las puertas de actualización ayudan a capturar dependencias a largo plazo en secuencias.

## Implementación desde cero
Para obtener una mejor comprensión del modelo GRU, vamos a implementarlo desde cero.

### Inicialización de los parámetros del modelo

El primer paso es inicializar los parámetros del modelo. Dibujamos los pesos de una distribución gaussiana con desviación estándar para ser `sigma` y establecemos el sesgo en 0. El hiperparametro `num_hiddens` define el número de unidades ocultas. Presentamos todos los pesos y sesgos relacionados con la compuerta de actualización, la compuerta de reinicio y el estado oculto candidato.


In [ ]:
class GRUScratch(d2l.Module):
    def __init__(self, num_inputs, num_hiddens, sigma=0.01):
        super().__init__()
        self.save_hyperparameters()

        init_weight = lambda *shape: nn.Parameter(torch.randn(*shape) * sigma)
        triple = lambda: (init_weight(num_inputs, num_hiddens),
                          init_weight(num_hiddens, num_hiddens),
                          nn.Parameter(torch.zeros(num_hiddens)))
        self.W_xz, self.W_hz, self.b_z = triple()  # Actualizar la puerta
        self.W_xr, self.W_hr, self.b_r = triple()  # Reiniciar la puerta
        self.W_xh, self.W_hh, self.b_h = triple()  # Estado oculto del candidato

### Definir el modelo
Ahora estamos listos para **definir el cálculo hacia adelante de GRU**. Su estructura es la misma que la de la célula RNN básica, excepto que las ecuaciones de actualización son más complejas.


In [ ]:
@d2l.add_to_class(GRUScratch)
def forward(self, inputs, H=None):
    if H is None:
        # Estado inicial con forma: (batch_size, num_hiddens)
        H = torch.zeros((inputs.shape[1], self.num_hiddens),
                      device=inputs.device)
    outputs = []
    for X in inputs:
        Z = torch.sigmoid(torch.matmul(X, self.W_xz) +
                        torch.matmul(H, self.W_hz) + self.b_z)
        R = torch.sigmoid(torch.matmul(X, self.W_xr) +
                        torch.matmul(H, self.W_hr) + self.b_r)
        H_tilde = torch.tanh(torch.matmul(X, self.W_xh) +
                           torch.matmul(R * H, self.W_hh) + self.b_h)
        H = Z * H + (1 - Z) * H_tilde
        outputs.append(H)
    return outputs, H

### Entrenamiento
**Entrenando** un modelo de lenguaje en *El conjunto de datos de Time Machine* funciona exactamente de la misma manera que en [Referencia sec_rnn-scratch](https://d2l.ai/chapter_recurrent-neural-networks/rnn-scratch.html#sec-rnn-scratch).


In [ ]:
data = d2l.TimeMachine(batch_size=1024, num_steps=32)
gru = GRUScratch(num_inputs=len(data.vocab), num_hiddens=32)
model = d2l.RNNLMScratch(gru, vocab_size=len(data.vocab), lr=4)
trainer = d2l.Trainer(max_epochs=50, gradient_clip_val=1, num_gpus=1)
trainer.fit(model, data)

### Nota docente de Hespérides

Escribe qué información puede ver cada posición. Una máscara causal impide consultar el futuro; una máscara de padding excluye posiciones que no son datos. Comprueba que cada fila de atención suma uno antes de aplicar dropout. Los mapas de atención describen mezclas de valores, pero por sí solos no prueban una explicación causal del modelo.

Vínculo con los apuntes: sesión 5, «GRU».


## Implementación concisa

En APIs de alto nivel, podemos instanciar directamente un modelo GRU. Esto encapsula todo el detalle de configuración que hicimos explícito arriba.


In [ ]:
class GRU(d2l.RNN):
    def __init__(self, num_inputs, num_hiddens):
        d2l.Module.__init__(self)
        self.save_hyperparameters()
        self.rnn = nn.GRU(num_inputs, num_hiddens)

El código es significativamente más rápido en el entrenamiento ya que utiliza operadores compilados en lugar de Python.


In [ ]:
gru = GRU(num_inputs=len(data.vocab), num_hiddens=32)
model = d2l.RNNLM(gru, vocab_size=len(data.vocab), lr=4)
trainer.fit(model, data)

Después del entrenamiento, imprimimos la perplejidad en el conjunto de entrenamiento y la secuencia predicha siguiendo el prefijo proporcionado.


In [ ]:
model.predict('it has', 20, data.vocab, d2l.try_gpu())

## Resumen
En comparación con los LSTM, los GRUs logran un rendimiento similar pero tienden a ser más ligeros computacionalmente. Generalmente, en comparación con los RNNs simples, los RNNS cerrados, al igual que los LSTMs y los GRUs, pueden capturar mejor las dependencias para secuencias con grandes distancias de paso. Los GRUs contienen RNNs básicos como su caso extremo cada vez que la compuerta de reinicio está activada.

## Ejercicios
1. Supongamos que sólo queremos utilizar la entrada en el paso de tiempo $t'$ para predecir la salida en el paso de tiempo $t > t'$. ¿Cuáles son los mejores valores para las puertas de reinicio y actualización para cada paso de tiempo?
1. Ajuste los hiperparametros y analice su influencia en el tiempo de ejecución, la perplejidad y la secuencia de salida.
1. Compare el tiempo de ejecución, la perplejidad y las cadenas de salida para las implementaciones `rnn.RNN` y `rnn.GRU` entre sí.
1. ¿Qué sucede si sólo implementa partes de un GRU, por ejemplo, con sólo una compuerta de reinicio o sólo una compuerta de actualización?


[Debate del original](https://discuss.d2l.ai/t/1056)
